In [1]:
"""
AgroSense — OpenWeatherMap API → BigQuery
==========================================
Converted from MySQL → BigQuery (Free Tier Compatible)

Fetches 5-day / 3-hour forecasts for every region in dim_regions
and appends the results to fact_weather_api in BigQuery.

Setup:
    pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

Authentication (run once in terminal):
    gcloud auth application-default login

API Key — set as environment variable (never hardcode):
    Windows : set OPENWEATHER_API_KEY=your_key_here
    Mac/Linux: export OPENWEATHER_API_KEY=your_key_here
"""

# ── STEP 1: Imports ────────────────────────────────────────────────────────────
import pandas as pd
import requests
import os
from google.cloud import bigquery
from datetime import datetime, timezone ,UTC

# ── STEP 2: Configuration ──────────────────────────────────────────────────────
PROJECT_ID      = "agrosense-493415"   # ← Replace with your GCP project ID
DATASET_ID      = "agrosense"
KEY_FILE     = "agrosense_key.json" 

# Read API key from environment variable — keep secrets out of code
#WEATHER_API_KEY = os.environ.get("OPENWEATHER_API_KEY", "your_api_key_here")



# ── STEP 3: BigQuery Client ────────────────────────────────────────────────────
# No password needed — uses Google Cloud credentials automatically.
# If using a service account key file, uncomment the line below:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_FILE

client = bigquery.Client(project=PROJECT_ID)
print("BigQuery client initialised — project:", PROJECT_ID)

# ── HELPERS ───────────────────────────────────────────────────────────────────
def get_next_id(table_name: str, id_column: str) -> int:
    try:
        result = client.query(f"""
            SELECT COALESCE(MAX({id_column}), 0) AS max_id
            FROM `{PROJECT_ID}.{DATASET_ID}.{table_name}`
        """).to_dataframe()
        return int(result["max_id"].iloc[0]) + 1
    except Exception:
        return 1


def bq_append(df: pd.DataFrame, table_name: str):
    table_id   = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    print(f"  ✓ Loaded {len(df)} rows → {table_id}")

print("✓ Helpers loaded!")

# ── STEP 1: Read dim_regions ──────────────────────────────────────────────────
print("\n  Reading dim_regions...")
dim_regions_df = client.query(f"""
    SELECT region_id, region_name, latitude, longitude
    FROM `{PROJECT_ID}.{DATASET_ID}.dim_regions`
""").to_dataframe()
print(f"  ✓ {len(dim_regions_df)} regions loaded")

# ── STEP 2: Fetch 5-day forecast ──────────────────────────────────────────────
print("\n  Fetching 5-day forecasts...")
forecast_list = []

for _, row in dim_regions_df.iterrows():
    print(f"\n    Region : {row['region_name']}")

    # ── 5-day forecast — 40 slots of 3-hour intervals ──────
    forecast_url = (
        f"https://api.openweathermap.org/data/2.5/forecast"
        f"?lat={row['latitude']}&lon={row['longitude']}"
        f"&appid={WEATHER_API_KEY}&units=metric"
    )
    weather_response = requests.get(forecast_url).json()

    # ── UV index — separate free call ──────────────────────
    uv_url = (
        f"https://api.openweathermap.org/data/2.5/uvi"
        f"?lat={row['latitude']}&lon={row['longitude']}"
        f"&appid={WEATHER_API_KEY}"
    )
    uv_response = requests.get(uv_url).json()
    uv_index    = uv_response.get("value", None)

    # ── Safety check ───────────────────────────────────────
    if "list" not in weather_response:
        print(f"    WARNING — {weather_response.get('message', 'unknown')}")
        continue

    print(f"    ✓ {len(weather_response['list'])} forecast slots | UV: {uv_index}")

    # ── Loop through all 40 time slots ─────────────────────
    for forecast in weather_response["list"]:
        forecast_list.append({
            "region_id"    : row["region_id"],
            "region_name"  : row["region_name"],
            "fetch_ts"     : forecast.get("dt_txt"),        # "2024-01-18 06:00:00"
            "temperature_C": forecast.get("main", {}).get("temp"),
            "humidity_pct" : forecast.get("main", {}).get("humidity"),
            "wind_speed_ms": forecast.get("wind", {}).get("speed", 0),
            "rainfall_mm"  : forecast.get("rain", {}).get("3h", 0),  # 3h for /forecast
            "uv_index"     : uv_index,
            "api_lat"      : row["latitude"],
            "api_lon"      : row["longitude"],
        })

print(f"\n  ✓ Total rows collected: {len(forecast_list)}")
print(f"  Expected            : {len(dim_regions_df) * 40} rows (40 slots × {len(dim_regions_df)} regions)")

# ── STEP 3: Build DataFrame ───────────────────────────────────────────────────
weather_df = pd.DataFrame(forecast_list)
print(f"\n  Columns : {weather_df.columns.tolist()}")
print(f"  Shape   : {weather_df.shape}")

# ── STEP 4: Fix types ─────────────────────────────────────────────────────────
# fetch_ts — string "2024-01-18 06:00:00" → BigQuery TIMESTAMP
weather_df["fetch_ts"] = pd.to_datetime(
    weather_df["fetch_ts"], utc=True
).astype("datetime64[us, UTC]")

weather_df["region_id"]     = weather_df["region_id"].astype("Int64")
weather_df["region_name"]   = weather_df["region_name"].astype(str)
weather_df["temperature_C"] = weather_df["temperature_C"].astype(float)
weather_df["humidity_pct"]  = weather_df["humidity_pct"].astype(float)
weather_df["wind_speed_ms"] = weather_df["wind_speed_ms"].astype(float)
weather_df["rainfall_mm"]   = weather_df["rainfall_mm"].astype(float)
weather_df["uv_index"]      = pd.to_numeric(
    weather_df["uv_index"], errors="coerce"
)
weather_df["api_lat"]       = weather_df["api_lat"].astype(float)
weather_df["api_lon"]       = weather_df["api_lon"].astype(float)

# ── STEP 5: Auto increment ────────────────────────────────────────────────────
next_id = get_next_id("fact_weather_api", "weather_id")
weather_df.insert(0, "weather_id", range(next_id, next_id + len(weather_df)))
weather_df["weather_id"] = weather_df["weather_id"].astype("Int64")

# ── STEP 6: ingested_at ───────────────────────────────────────────────────────
weather_df["ingested_at"] = pd.Timestamp.now(tz="UTC")
weather_df["ingested_at"] = pd.to_datetime(
    weather_df["ingested_at"], utc=True
).astype("datetime64[us, UTC]")

# ── STEP 7: Final column order ────────────────────────────────────────────────
weather_df = weather_df[[
    "weather_id",
    "region_id",
    "region_name",
    "fetch_ts",
    "temperature_C",
    "humidity_pct",
    "wind_speed_ms",
    "rainfall_mm",
    "uv_index",
    "api_lat",
    "api_lon",
    "ingested_at"
]]

# ── STEP 8: Preview ───────────────────────────────────────────────────────────
print("\n  Sample rows:")
print(weather_df.head(5).to_string(index=False))
print("\n  Data types:")
print(weather_df.dtypes)

# ── STEP 9: Upload to BigQuery ────────────────────────────────────────────────
bq_append(weather_df, "fact_weather_api")
print("\n  ✓ 5-day forecast loaded successfully!")

# ── STEP 10: Verify ───────────────────────────────────────────────────────────
result = client.query(f"""
    SELECT
        COUNT(*)                            AS total_rows,
        MIN(fetch_ts)                       AS earliest_forecast,
        MAX(fetch_ts)                       AS latest_forecast,
        MAX(ingested_at)                    AS last_ingested,
        COUNT(DISTINCT DATE(fetch_ts))      AS forecast_days,
        COUNT(DISTINCT region_name)         AS regions
    FROM `{PROJECT_ID}.{DATASET_ID}.fact_weather_api`
""").to_dataframe()

print("\n  Verification:")
print(result.to_string(index=False))

BigQuery client initialised — project: agrosense-493415
✓ Helpers loaded!

  Reading dim_regions...
  ✓ 5 regions loaded

  Fetching 5-day forecasts...

    Region : North India
    ✓ 40 forecast slots | UV: 9.7

    Region : South India
    ✓ 40 forecast slots | UV: 13.86

    Region : Central USA
    ✓ 40 forecast slots | UV: 6.28

    Region : South USA
    ✓ 40 forecast slots | UV: 8.58

    Region : East Africa
    ✓ 40 forecast slots | UV: 13.47

  ✓ Total rows collected: 200
  Expected            : 200 rows (40 slots × 5 regions)

  Columns : ['region_id', 'region_name', 'fetch_ts', 'temperature_C', 'humidity_pct', 'wind_speed_ms', 'rainfall_mm', 'uv_index', 'api_lat', 'api_lon']
  Shape   : (200, 10)

  Sample rows:
 weather_id  region_id region_name                  fetch_ts  temperature_C  humidity_pct  wind_speed_ms  rainfall_mm  uv_index  api_lat  api_lon                      ingested_at
       1811          1 North India 2026-04-21 09:00:00+00:00          38.06          16

KeyboardInterrupt: 